# Volet 1 — Pipeline HTR (renforcé) · CATMuS Medieval français

Notebook **Colab / A100** qui s'appuie sur les modules `src/htr/` du dépôt
(`git clone` + `pip install -e .`) plutôt que de redéfinir le code dans le notebook.
Objectif : **combler les écarts slides ↔ code** identifiés à l'audit —

1. **pytest vert** (preuve sur A100) ;
2. **prétraitement réel** : deskew + CLAHE + Sauvola ;
3. **segmentation** projection *et* Kraken BLLA + **mesure d'IoU** + export **PAGE XML** ;
4. **éval test scellé** : CER/WER + **IC bootstrap** ;
5. **comparaison r=16 vs r=32** (augmentation) avec **test de McNemar** ;
6. **data contract** `dataset_nlp.json` **validé par jsonschema** ;
7. **journal d'expériences** `experiments/journal.jsonl`.

> Reproductibilité : `seed=42` partout, pile figée via l'extra `[train]` du pyproject.

## 0. Installation du dépôt + vérification environnement

In [ ]:
# Cloner le dépôt (ou mettre à jour s'il existe déjà) et installer le package.
import os
REPO = "ComputerVision_NLP_HETIC_MD5"
URL  = "https://github.com/FaridBenamara/ComputerVision_NLP_HETIC_MD5.git"
if not os.path.exists(REPO):
    !git clone -q $URL
else:
    !cd $REPO && git pull -q
%cd $REPO
# [dev] = cœur CPU + pytest (numpy, opencv, scikit-image, jsonschema, lxml, editdistance, jiwer).
!pip install -q -e ".[dev]"
print("Installé. Pour l'entraînement/inférence TrOCR :  pip install -q -e \".[train]\"")

In [ ]:
import sys, torch
from htr.seeds import fixer_seeds
fixer_seeds(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python :", sys.version.split()[0])
print("Device :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU")

## 1. Tests automatisés (pytest) — preuve que la base est saine

Exigence brief CV §5 / NLP §5. Doit afficher **tout vert**.

In [ ]:
!python -m pytest -q

## 2. Données — CATMuS Medieval, sous-ensemble français

Splits officiels (`gen_split`), aucune fuite. EDA courte + scellement SHA-256 du train.

In [ ]:
from htr import data
ds = data.charger_catmus_francais()            # télécharge les parquet L-Fre* (caché ensuite)
print("Tailles :", {sp: len(ds[sp]) for sp in ds})

import collections
print("Siècles  :", dict(sorted(collections.Counter(ds["train"]["century"]).items())))
print("Écritures:", collections.Counter(ds["train"]["script_type"]).most_common(5))

# Nettoyage + rapport avant/après (normalisation NFC, lignes vides retirées)
gardes, rapport = data.nettoyer_corpus(ds["train"]["text"], longueur_min=1)
print("Nettoyage train :", rapport)

# SHA-256 du train (anti-contamination) — à reporter, puis ne plus regarder le test
TRAIN_SHA = data.sha256_split(ds["train"]["text"], ds["train"]["shelfmark"])
print("train SHA-256 :", TRAIN_SHA)

## 3. Prétraitement réel — deskew · CLAHE · Sauvola

La brique annoncée dans les slides mais absente du 1er notebook. On illustre chaque
étape sur une page assemblée à partir de lignes du test (CATMuS fournit des lignes
pré-découpées, pas des pages entières).

In [ ]:
import numpy as np
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
from htr import preprocessing as P

# Page dense (>=18 lignes d'un même manuscrit français)
cnt = Counter(ds["test"]["shelfmark"])
cote = [s for s, n in cnt.most_common() if n >= 18][0]
idxs = [i for i in range(len(ds["test"])) if ds["test"][i]["shelfmark"] == cote][:18]
crops = [ds["test"][i]["im"].convert("RGB") for i in idxs]
gt_lignes = [data.nettoyer_texte(ds["test"][i]["text"]) for i in idxs]

W = max(c.width for c in crops) + 80; gap = 16
H = sum(c.height for c in crops) + gap * (len(crops) + 1)
page = Image.new("RGB", (W, H), (247, 244, 237)); y = gap
bornes_vraies = []                       # y0,y1 connus de chaque ligne -> référence IoU
for c in crops:
    page.paste(c, (40, y)); bornes_vraies.append((y, y + c.height)); y += c.height + gap
page_np = np.array(page)
print("Manuscrit :", cote, "| page", page.size, "|", len(crops), "lignes")

res = P.pretraiter(page_np, deskew=True, clahe=True, sauvola=True)
print("Inclinaison corrigée :", res["angle"], "°")

fig, ax = plt.subplots(1, 4, figsize=(18, 8))
for a, k in zip(ax, ["gris", "deskew", "clahe", "sauvola"]):
    a.imshow(res["etapes"][k], cmap="gray"); a.set_title(k); a.axis("off")
plt.tight_layout(); plt.show()

## 4. Segmentation + IoU + PAGE XML

On segmente la page (projection, et Kraken BLLA si installé), on **mesure l'IoU**
contre les positions vraies des lignes, et on **exporte en PAGE XML**.

In [ ]:
from htr import segmentation as S, metrics as M, page_xml as PX

lignes = S.segmenter_par_projection(page_np)
print("Lignes détectées (projection) :", len(lignes), "| vraies :", len(bornes_vraies))

# IoU : pour chaque ligne vraie, meilleure IoU avec une ligne prédite (proxy de segmentation)
def poly_bande(y0, y1, w): return [(0, y0), (w - 1, y0), (w - 1, y1), (0, y1)]
ref_polys  = [poly_bande(y0, y1, W) for (y0, y1) in bornes_vraies]
pred_polys = [l["poly"] for l in lignes]
ious = [max((M.iou_polygones(r, p) for p in pred_polys), default=0.0) for r in ref_polys]
print(f"IoU moyenne (projection vs vérité) : {np.mean(ious):.3f}  (seuil brief > 0.75)")

# Kraken BLLA (optionnel : pip install -e ".[kraken]")
try:
    page.save("/content/_page_demo.jpg")
    lignes_kraken = S.segmenter_kraken_blla("/content/_page_demo.jpg")
    print("Lignes détectées (Kraken BLLA) :", len(lignes_kraken))
except ImportError as e:
    print("Kraken non installé — fallback projection. (", e, ")")

# Export PAGE XML
xml = PX.exporter_page_xml(lignes, "page_demo.jpg", largeur=W, hauteur=H)
os.makedirs("segmentations", exist_ok=True)
open("segmentations/page_demo.page.xml", "wb").write(xml)
print("PAGE XML écrit -> segmentations/page_demo.page.xml")

## 5. Modèle r=16 — chargement depuis Drive + éval test scellé + IC bootstrap

> Nécessite la pile d'entraînement et le modèle entraîné sur ton Drive.
> Exécute d'abord la cellule d'install `[train]` ci-dessous (un redémarrage Colab
> peut être demandé : relance ensuite à partir d'ici).

In [ ]:
# Pile figée pour TrOCR+LoRA (ne réinstalle PAS torch -> garde le build GPU).
!pip install -q -e ".[train]"
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/htr-catmus-french-2026"
ADAPTER  = SAVE_DIR + "/trocr-fr-lora-final"
print("Adaptateur attendu :", ADAPTER, "| existe :", os.path.exists(ADAPTER))

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from peft import PeftModel
from htr import model as Mh, metrics as M, data

def charger_modele_lora(adapter_dir, base_id="microsoft/trocr-base-handwritten"):
    processor = TrOCRProcessor.from_pretrained(adapter_dir)
    base = VisionEncoderDecoderModel.from_pretrained(base_id)
    tok = processor.tokenizer
    for cfg in (base.config, base.generation_config):
        cfg.decoder_start_token_id = tok.cls_token_id
        cfg.pad_token_id = tok.pad_token_id
        cfg.eos_token_id = tok.sep_token_id
    base.config.vocab_size = base.config.decoder.vocab_size
    model = PeftModel.from_pretrained(base, adapter_dir).to(DEVICE).eval()
    return model, processor

model, processor = charger_modele_lora(ADAPTER)

# Éval sur le test scellé (échantillon n=500 fixé par seed ; passe à len(ds["test"]) pour le test complet)
N = 500
test = ds["test"].shuffle(seed=42).select(range(N))
refs  = [data.nettoyer_texte(t) for t in test["text"]]
preds = Mh.transcrire(model, processor, test["im"], device=DEVICE, batch_size=16)

ic = M.bootstrap_ic_cer(refs, preds, n_resamples=1000, seed=42)
print(f"=== TEST SCELLÉ (n={N}) ===")
print(f"CER {ic['cer']*100:.2f}%  IC95%[{ic['ic_bas']*100:.2f},{ic['ic_haut']*100:.2f}]  WER {M.wer(refs,preds)*100:.2f}%")

## 6. Run 2 — r=32 + augmentation, puis comparaison McNemar (r=16 vs r=32)

Le 1er notebook avait échoué ici (`ModuleNotFoundError: jiwer`). Ici les deps sont
gérées par le package. Entraînement long (~50 min A100) — cellule **optionnelle**.

In [ ]:
# ----- Entraînement r=32 + augmentation (optionnel, long) -----
import cv2
from PIL import Image
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from htr import model as Mh

fixer_seeds(42)
model32, processor32 = Mh.construire_trocr_lora(r=32, lora_alpha=64)
tok32 = processor32.tokenizer
PAD = tok32.pad_token_id

_rng = np.random.default_rng(42)
def augment(pil):
    a = np.array(pil.convert("RGB"))
    if _rng.random() < 0.5:
        a = np.clip((1 + _rng.uniform(-0.3, 0.3)) * a + _rng.uniform(-25, 25), 0, 255).astype(np.uint8)
    if _rng.random() < 0.5:
        h, w = a.shape[:2]; Mr = cv2.getRotationMatrix2D((w/2, h/2), _rng.uniform(-2, 2), 1.0)
        a = cv2.warpAffine(a, Mr, (w, h), borderMode=cv2.BORDER_REPLICATE)
    if _rng.random() < 0.4:
        a = np.clip(a + _rng.normal(0, 8, a.shape), 0, 255).astype(np.uint8)
    return Image.fromarray(a)

def collate_aug(batch):
    imgs = [augment(e["im"]) for e in batch]; txt = [e["text"] for e in batch]
    pv = processor32(images=imgs, return_tensors="pt").pixel_values
    lab = tok32(txt, padding=True, return_tensors="pt").input_ids; lab[lab == PAD] = -100
    return {"pixel_values": pv, "labels": lab}

val_small = ds["validation"].shuffle(seed=42).select(range(300))
args = Seq2SeqTrainingArguments(
    output_dir="/content/ckpts_r32", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    learning_rate=1e-4, warmup_ratio=0.05, eval_strategy="steps", eval_steps=1500,
    save_strategy="no", logging_steps=100, predict_with_generate=True,
    generation_max_length=128, bf16=True, report_to="none", remove_unused_columns=False)
trainer = Seq2SeqTrainer(model=model32, args=args, train_dataset=ds["train"],
                         eval_dataset=val_small, data_collator=collate_aug,
                         compute_metrics=Mh.metrics_factory(tok32))
trainer.train()
Mh.sauver_adaptateurs(model32, processor32, SAVE_DIR + "/trocr-fr-lora-r32")
print("r=32 sauvegardé ->", SAVE_DIR + "/trocr-fr-lora-r32")

In [ ]:
# ----- Comparaison r=16 vs r=32 sur le même test (test de McNemar) -----
from htr import metrics as M
model32, processor32 = charger_modele_lora(SAVE_DIR + "/trocr-fr-lora-r32")
preds32 = Mh.transcrire(model32, processor32, test["im"], device=DEVICE, batch_size=16)

print(f"r=16  CER {M.cer(refs,preds)*100:.2f}%  WER {M.wer(refs,preds)*100:.2f}%")
print(f"r=32  CER {M.cer(refs,preds32)*100:.2f}%  WER {M.wer(refs,preds32)*100:.2f}%")
mc = M.mcnemar(refs, preds, preds32)
print("McNemar :", mc)
print("Significatif (p<0.05) :", mc["p_value"] < 0.05)

## 7. Data contract — génération + validation jsonschema + calibration du seuil

In [ ]:
from htr import data_contract as DC, metrics as M, data
import editdistance

SUBSET = ds["test"].select(range(300))
rich = Mh.transcrire_riche(model, processor, SUBSET["im"], device=DEVICE, batch_size=8)

lignes = []
for idx, (ex, r) in enumerate(zip(SUBSET, rich)):
    w, h_ = ex["im"].size
    lignes.append(DC.construire_ligne(
        idx=idx, texte=r["texte"],
        polygone=[[0, 0], [w, 0], [w, h_], [0, h_]],   # bbox du crop (ligne pré-découpée)
        confiance=r["conf"], confiance_char=r["conf_char"], candidats_char=r["cand_char"],
        image_source=ex["shelfmark"],
        metadonnees={"siecle": ex["century"], "script_type": ex["script_type"]},
        seuil_needs_review=0.70))

doc = DC.construire_document(lignes, corpus="CATMuS Medieval (français)",
                             modele="trocr-fr-lora (r=16)", train_sha256=TRAIN_SHA)
DC.valider(doc)                                  # lève si non conforme au schéma
chemin = DC.ecrire(doc, "dataset_nlp/dataset_nlp.json")
tx = sum(l["needs_review"] for l in lignes) / len(lignes)
print(f"dataset_nlp.json validé ✅  ({len(lignes)} lignes) | needs_review = {tx*100:.1f}% -> {chemin}")

# Calibration du seuil (corrélation confiance <-> erreur)
refs_sub = [data.nettoyer_texte(t) for t in SUBSET["text"]]
confs = np.array([l["confiance"] for l in lignes])
cers  = M.cer_par_ligne(refs_sub, [l["texte"] for l in lignes])
print(f"Corrélation conf/CER : {np.corrcoef(confs, cers)[0,1]:.3f}")
for q in [0.60, 0.65, 0.70, 0.75, 0.80]:
    flag = confs < q
    print(f"  seuil {q:.2f} -> review {flag.mean()*100:4.1f}%  CER_flag {cers[flag].mean()*100 if flag.any() else 0:5.1f}%  CER_ok {cers[~flag].mean()*100:5.1f}%")

## 8. Journal d'expériences (`experiments/journal.jsonl`)

In [ ]:
import json, datetime
os.makedirs("experiments", exist_ok=True)
entree = {
    "run": "volet1_renforce",
    "seed": 42,
    "modele": "trocr-base-handwritten + LoRA r=16 (alpha=32)",
    "cer_test": round(ic["cer"], 4),
    "ic95": [round(ic["ic_bas"], 4), round(ic["ic_haut"], 4)],
    "wer_test": round(M.wer(refs, preds), 4),
    "n_test": N,
    "needs_review": round(tx, 4),
    "train_sha256": TRAIN_SHA,
    "horodatage": datetime.datetime.utcnow().isoformat() + "Z",
}
with open("experiments/journal.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(entree, ensure_ascii=False) + "\n")
print("Journalisé :", entree)

## Récap

| Élément | Statut |
|---|---|
| pytest | vert (cellule 1) |
| Prétraitement deskew/CLAHE/Sauvola | implémenté + illustré |
| Segmentation + IoU + PAGE XML | mesuré + exporté |
| CER/WER + IC bootstrap | test scellé |
| r=16 vs r=32 + McNemar | comparé |
| data contract validé jsonschema | `dataset_nlp/dataset_nlp.json` |
| journal d'expériences | `experiments/journal.jsonl` |

**Pense à committer/pousser** les artefacts légers générés (`dataset_nlp/`,
`segmentations/*.page.xml`, `experiments/journal.jsonl`). Les modèles lourds vont
sur HuggingFace (lien dans le README), pas dans git.